# MEK-Means Rust Fast-Paths — PBMC 10k Demo

This notebook demonstrates and benchmarks the Rust acceleration of the
**MEK-Means** (Mixture-of-Experts K-Means) EM algorithm in `mminference.py`.

Dataset: **PBMC 10k** (10 997 cells, 3 known cell types: T cells, Monocytes, B cells).  
We fit k=3 mixture components and compare recovered clusters to ground-truth labels.

| Rust function | Replaces | What it accelerates |
|---|---|---|
| `e_step_2d` | Python loop in `_e_step` | Parallel PSS eval over k × genes; no GIL |
| `optimize_genes_2d` | `scipy.optimize.minimize` loop in `iterate_over_genes` | Parallel L-BFGS-B over all genes; no GIL |

Both activate automatically when `bio_model ∈ {Bursty, CIR, …}` and `hist_type=unique`.

## Setup

In [ ]:
import importlib, subprocess, sys, os, pathlib

def _try_run(*args):
    env = os.environ.copy()
    extra = [str(pathlib.Path.home() / '.cargo' / 'bin'),
             str(pathlib.Path(sys.prefix) / 'bin')]
    env['PATH'] = os.pathsep.join(extra) + os.pathsep + env.get('PATH', '')
    try:
        return subprocess.check_output(list(args), stderr=subprocess.DEVNULL,
                                       text=True, env=env).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

spec = importlib.util.find_spec('monod_core')
if spec is None:
    print('monod_core NOT found.  Build with:')
    print('  cd monod_core && maturin develop --release')
    rust_ready = False
else:
    import monod_core as _mc
    print(f'monod_core : {_mc.__file__}')
    rustc = _try_run('rustc', '--version')
    print(f'rustc      : {rustc or "not found (ok — extension already built)"}')
    rust_ready = True
    print('\nRust extension ready.')

In [ ]:
import sys, os, time, warnings
import numpy as np
import scipy.sparse, scipy.optimize
import matplotlib.pyplot as plt
from scipy.special import softmax, logsumexp
from sklearn.metrics import adjusted_rand_score, confusion_matrix

_src = os.path.join(os.path.abspath('.'), 'src', 'monod')
if _src not in sys.path:
    sys.path.insert(0, _src)

from cme_toolbox import CMEModel, _HAS_RUST
import anndata as ad
from extract_data import extract_data
from inference import searchdata_from_adata, SearchData
from mminference import GradientInference, _mc as mm_mc, _HAS_RUST as mm_rust
import mminference as mm

warnings.filterwarnings('ignore')

C_PY   = 'steelblue'
C_RUST = 'firebrick'

print(f'_HAS_RUST (cme_toolbox) = {_HAS_RUST}')
print(f'_HAS_RUST (mminference) = {mm_rust}')

In [ ]:
# ── Load PBMC 10k dataset ────────────────────────────────────────────────────
adata = ad.read_h5ad('example_h5ad/processed_pbmc_10k_raw.h5ad')
adata.var_names_make_unique()

cell_types = adata.obs['subclass_label'].values
unique_types, type_counts = np.unique(cell_types, return_counts=True)
print(f'Dataset: {adata.n_obs:,} cells  |  {adata.n_vars:,} genes')
print('Cell types:')
for ct, n in sorted(zip(unique_types, type_counts), key=lambda x: -x[1]):
    print(f'  {ct:20s}: {n:5,} cells ({100*n/adata.n_obs:.1f}%)')

var_to_name = dict(zip(adata.var_names, adata.var['gene_name']))

In [ ]:
# ── Select expressed genes and build search data ─────────────────────────────
MODEL   = CMEModel('Bursty', 'None')
N_GENES = 25

s_mat = adata.layers['spliced']
if scipy.sparse.issparse(s_mat):
    s_mat = s_mat.toarray()
expressed = np.where((s_mat > 0).sum(0) >= 100)[0]
GENES = [adata.var_names[i] for i in expressed[:N_GENES]]
gene_display = [var_to_name[g] for g in GENES]
print(f'Selected {N_GENES} genes expressed in ≥100 cells')
print('Gene names:', gene_display)

t0 = time.perf_counter()
adata_ext = extract_data(
    adata, MODEL,
    dataset_name='/tmp/pbmc_mmdemo',
    modality_name_dict={'unspliced': 'unspliced', 'spliced': 'spliced'},
    n_genes=N_GENES, genes_to_fit=GENES,
    hist_type='unique', viz=False,
)
search_data = searchdata_from_adata(adata_ext)
print(f'\nSearchData ready in {time.perf_counter()-t0:.1f}s: '
      f'{search_data.n_genes} genes, {search_data.n_cells:,} cells')

---
## 1  E-step fast-path: `e_step_2d`

For each cell $c$ and cluster $k$ the E-step computes:
$$\log L_{c,k} = \sum_g \log P_{\theta_k^g}(u_c^g, s_c^g) + \log w_k$$

`e_step_2d` evaluates all $k \times G$ PSS grids in parallel (rayon) and
runs the softmax inside Rust — no GIL round-trips.

In [ ]:
# ── Helper: build a GradientInference without file I/O ───────────────────────
def make_gi(k, epochs, regressor):
    lb = np.array([-3., -3., -3.])
    ub = np.array([ 3.,  3.,  3.])
    g = GradientInference.__new__(GradientInference)
    g.gradient_params = {'max_iterations': 15, 'init_pattern': 'moments', 'num_restarts': 1}
    g.phys_lb  = lb
    g.phys_ub  = ub
    g.grad_bnd = scipy.optimize.Bounds(lb, ub)
    g.n_phys_pars = MODEL.get_num_params()
    g.n_samp_pars = 2
    g.inference_string = '/tmp/pbmc_mmdemo'
    g.k       = k
    g.epochs  = epochs
    g.weights = np.ones(k) / k
    g.theta   = {}
    g.regressor = regressor
    g.param_MoM = np.asarray([
        MODEL.get_MoM(search_data.moments[i], lb, ub, regressor[i])
        for i in range(search_data.n_genes)
    ])
    return g

K   = 3    # matches the 3 known PBMC cell types
reg = np.zeros((search_data.n_genes, 2))   # seq_model=None

# Run one M-step to get a plausible theta for correctness checking
gi = make_gi(K, epochs=1, regressor=reg)
Q_init = gi._initialize_Q(search_data)
k_dict = gi._part_search_data(search_data, Q_init)
gi._m_step(MODEL, k_dict, Q_init)
print(f'Initial M-step done. Clusters with data: {sorted(gi.theta.keys())}')
print(f'Params shape per cluster: {gi.theta[0][0].shape}')

In [ ]:
# ── Correctness: Rust vs Python E-step ───────────────────────────────────────
EPS = 1e-15
ks_present   = sorted(gi.theta.keys())
n_genes      = search_data.n_genes
n_cells      = search_data.n_cells
params_per_k = [gi.theta[k][0].tolist() for k in ks_present]
limits_list  = [[int(v) for v in search_data.M[:, g]] for g in range(n_genes)]
u_obs = [search_data.layers[0][:, g].astype(int).tolist() for g in range(n_genes)]
s_obs = [search_data.layers[1][:, g].astype(int).tolist() for g in range(n_genes)]
weights_sub  = gi.weights[ks_present].tolist()

# Rust
t0 = time.perf_counter()
Q_rust, lb_rust, _ = _mc.e_step_2d(
    bio_model='Bursty', params_per_k=params_per_k,
    limits_list=limits_list, u_obs=u_obs, s_obs=s_obs,
    weights=weights_sub,
    fixed_quad_t=float(MODEL.fixed_quad_T), quad_order=int(MODEL.quad_order),
    eps=EPS,
)
t_rust_e = time.perf_counter() - t0
Q_rust = np.array(Q_rust)

# Python reference
t0 = time.perf_counter()
logL_py = np.zeros((n_cells, len(ks_present)))
for ki, k in enumerate(ks_present):
    params, *_ = gi.theta[k]
    logL_k = np.zeros(n_cells)
    for g in range(n_genes):
        U = search_data.layers[0][:, g].astype(int)
        S = search_data.layers[1][:, g].astype(int)
        pss = MODEL.eval_model_pss(params[g], search_data.M[:, g], None)
        pss[pss < EPS] = EPS
        logL_k += np.log(pss[U, S])
    logL_py[:, ki] = logL_k
logL_py += np.log(np.array(weights_sub))[None, :]
Q_py = softmax(logL_py, axis=1)
t_py_e = time.perf_counter() - t0

max_diff = np.abs(Q_rust - Q_py).max()
print(f'Max |Q_rust − Q_python| = {max_diff:.2e}  (should be ~0)')
print(f'E-step  Rust={t_rust_e*1e3:.0f} ms  Python={t_py_e*1e3:.0f} ms  ({t_py_e/t_rust_e:.1f}×)')
assert max_diff < 1e-8

In [ ]:
# ── Benchmark: E-step timing vs n_genes ──────────────────────────────────────
N_REPS = 3
gene_counts = [5, 10, 15, 20, 25]
rust_e_times, py_e_times = [], []

for ng in gene_counts:
    p_k  = [params_per_k[0][:ng]] * K
    lims = limits_list[:ng]
    u_o  = u_obs[:ng]
    s_o  = s_obs[:ng]
    wts  = [1.0 / K] * K

    t0 = time.perf_counter()
    for _ in range(N_REPS):
        _mc.e_step_2d('Bursty', p_k, lims, u_o, s_o, wts,
                      float(MODEL.fixed_quad_T), int(MODEL.quad_order), eps=EPS)
    rust_e_times.append((time.perf_counter() - t0) / N_REPS * 1e3)

    t0 = time.perf_counter()
    for _ in range(N_REPS):
        logL_b = np.zeros((n_cells, K))
        for ki in range(K):
            logL_k = np.zeros(n_cells)
            for g in range(ng):
                pss = MODEL.eval_model_pss(np.array(p_k[ki][g]), np.array(lims[g]), None)
                pss[pss < EPS] = EPS
                logL_k += np.log(pss[np.array(u_o[g]), np.array(s_o[g])])
            logL_b[:, ki] = logL_k
        logL_b += np.log(np.array(wts))[None, :]
        _ = softmax(logL_b, axis=1)
    py_e_times.append((time.perf_counter() - t0) / N_REPS * 1e3)

    print(f'  n_genes={ng:2d}  Python={py_e_times[-1]:7.0f} ms  Rust={rust_e_times[-1]:5.0f} ms  ({py_e_times[-1]/rust_e_times[-1]:.1f}×)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(gene_counts, py_e_times,   'o-', color=C_PY,   label='Python (GIL-bound loop)')
ax.plot(gene_counts, rust_e_times, 's-', color=C_RUST,  label='Rust (rayon parallel)')
ax.set_xlabel('Number of genes')
ax.set_ylabel('E-step wall time (ms)')
ax.set_title(f'E-step timing: k={K} clusters, {n_cells:,} cells')
ax.legend()
ax2 = ax.twinx()
speedups_e = [p/r for p, r in zip(py_e_times, rust_e_times)]
ax2.plot(gene_counts, speedups_e, '^--', color='darkgreen', alpha=0.6)
ax2.set_ylabel('Speedup (×)', color='darkgreen')
ax2.tick_params(axis='y', labelcolor='darkgreen')
fig.tight_layout()
plt.savefig('figures/pbmc_estep_bench.png', dpi=150)
plt.show()
print(f'Peak E-step speedup: {max(speedups_e):.1f}× at {gene_counts[np.argmax(speedups_e)]} genes')

---
## 2  M-step fast-path: `optimize_genes_2d`

`iterate_over_genes` calls the same Rust L-BFGS-B used in `inference.py`:
all genes are optimised in parallel under rayon — no GIL.

In [ ]:
# ── Correctness: Rust M-step vs scipy ────────────────────────────────────────
k_dict_check = gi._part_search_data(search_data, Q_init)
first_k = list(k_dict_check.keys())[0]
sd_k = k_dict_check[first_k]
print(f'Cluster {first_k}: {sd_k.n_cells:,} cells')

t0 = time.perf_counter()
params_rust, klds_rust, obj_rust, _ = gi.iterate_over_genes(MODEL, sd_k)
t_rust_m = time.perf_counter() - t0

mm._HAS_RUST = False
t0 = time.perf_counter()
params_py_m, klds_py_m, obj_py_m, _ = gi.iterate_over_genes(MODEL, sd_k)
t_py_m = time.perf_counter() - t0
mm._HAS_RUST = True

ratio = klds_rust / klds_py_m.clip(1e-15)
print(f'KLD ratio Rust/scipy  min={ratio.min():.3f}  median={np.median(ratio):.3f}  max={ratio.max():.3f}')
print(f'Total KLD  Rust={obj_rust:.3f}  scipy={obj_py_m:.3f}')
print(f'Wall time  Rust={t_rust_m*1e3:.0f} ms  scipy={t_py_m*1e3:.0f} ms  ({t_py_m/t_rust_m:.1f}×)')

In [ ]:
# ── Benchmark: M-step timing vs n_genes ──────────────────────────────────────
gene_counts_m = [5, 10, 15, 20, 25]
rust_m_times, py_m_times = [], []

def subset_sd(sd, n):
    attrs = ['M','hist','moments','n_genes','gene_names','n_cells','layers','hist_type','layer_names']
    vals  = [sd.M[:,:n], sd.hist[:n], sd.moments[:n], n, sd.gene_names[:n],
             sd.n_cells, sd.layers[:,:,:n], sd.hist_type, sd.layer_names]
    return SearchData(attrs, *vals)

gi_b = make_gi(K, epochs=1, regressor=reg)

for ng in gene_counts_m:
    sd_sub = subset_sd(sd_k, ng)
    gi_b.regressor = reg[:ng]
    gi_b.param_MoM = gi.param_MoM[:ng]

    t0 = time.perf_counter()
    gi_b.iterate_over_genes(MODEL, sd_sub)
    rust_m_times.append((time.perf_counter() - t0) * 1e3)

    mm._HAS_RUST = False
    t0 = time.perf_counter()
    gi_b.iterate_over_genes(MODEL, sd_sub)
    py_m_times.append((time.perf_counter() - t0) * 1e3)
    mm._HAS_RUST = True

    print(f'  n_genes={ng:2d}  scipy={py_m_times[-1]:7.0f} ms  Rust={rust_m_times[-1]:5.0f} ms  ({py_m_times[-1]/rust_m_times[-1]:.1f}×)')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(gene_counts_m, py_m_times,   'o-', color=C_PY,   label='scipy L-BFGS-B (sequential)')
ax.plot(gene_counts_m, rust_m_times, 's-', color=C_RUST,  label='Rust L-BFGS-B (rayon parallel)')
ax.set_xlabel('Number of genes')
ax.set_ylabel('M-step wall time (ms)')
ax.set_title(f'M-step timing: 15 iterations, {sd_k.n_cells:,} cells')
ax.legend()
speedups_m = [p/r for p, r in zip(py_m_times, rust_m_times)]
ax2 = ax.twinx()
ax2.plot(gene_counts_m, speedups_m, '^--', color='darkgreen', alpha=0.6)
ax2.set_ylabel('Speedup (×)', color='darkgreen')
ax2.tick_params(axis='y', labelcolor='darkgreen')
fig.tight_layout()
plt.savefig('figures/pbmc_mstep_bench.png', dpi=150)
plt.show()
print(f'Peak M-step speedup: {max(speedups_m):.1f}× at {gene_counts_m[np.argmax(speedups_m)]} genes')

---
## 3  Full EM — k=3 on PBMC 10k

Run MEK-Means end-to-end with both Rust fast-paths and compare to the Python
fallback. Then evaluate cluster assignments against known cell-type labels.

In [ ]:
K_EM     = 3
N_EPOCHS = 6

print(f'Running MEK-Means: k={K_EM}, epochs={N_EPOCHS}, '
      f'{search_data.n_genes} genes, {search_data.n_cells:,} cells')

# Rust fast-paths
gi_em = make_gi(K_EM, N_EPOCHS, reg)
Q0 = gi_em._initialize_Q(search_data)
kd0 = gi_em._part_search_data(search_data, Q0)
gi_em._m_step(MODEL, kd0, Q0)

t0 = time.perf_counter()
Q_em, _, all_qs_rust, _ = gi_em._fit(MODEL, search_data)
t_rust_em = time.perf_counter() - t0
print(f'\nRust  EM: {t_rust_em:.1f}s  |  final Q-func: {all_qs_rust[-1]:.1f}')

# Python fallback
mm._HAS_RUST = False
gi_py_em = make_gi(K_EM, N_EPOCHS, reg)
Q0p = gi_py_em._initialize_Q(search_data)
kd0p = gi_py_em._part_search_data(search_data, Q0p)
gi_py_em._m_step(MODEL, kd0p, Q0p)

t0 = time.perf_counter()
Q_py_em, _, all_qs_py, _ = gi_py_em._fit(MODEL, search_data)
t_py_em = time.perf_counter() - t0
mm._HAS_RUST = True

print(f'Python EM: {t_py_em:.1f}s  |  final Q-func: {all_qs_py[-1]:.1f}')
print(f'\nEnd-to-end speedup: {t_py_em / t_rust_em:.1f}×')

In [ ]:
# ── Convergence + cluster size plots ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epochs_x = np.arange(1, N_EPOCHS + 1)
ax = axes[0]
ax.plot(epochs_x, all_qs_rust, 'o-', color=C_RUST, label=f'Rust ({t_rust_em:.1f}s)')
ax.plot(epochs_x, all_qs_py,   's--', color=C_PY,  label=f'Python ({t_py_em:.1f}s)')
ax.set_xlabel('Epoch')
ax.set_ylabel('EM Q-function')
ax.set_title('Convergence')
ax.legend()

assigns = np.argmax(Q_em, axis=1)
unique_k, counts_k = np.unique(assigns, return_counts=True)
ax = axes[1]
bars = ax.bar([f'Cluster {u}' for u in unique_k],
              counts_k, color=plt.cm.tab10(np.linspace(0, 0.3, K_EM)))
for b, c in zip(bars, counts_k):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 50,
            f'{c:,}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Number of cells')
ax.set_title(f'Cluster sizes (k={K_EM})')
fig.tight_layout()
plt.savefig('figures/pbmc_em_convergence.png', dpi=150)
plt.show()

In [ ]:
# ── Compare MEK-Means clusters to ground-truth cell types ────────────────────
obs_names_ext = list(adata_ext.obs_names)
obs_names_all = list(adata.obs_names)
kept_idx = [obs_names_all.index(n) for n in obs_names_ext]
cell_types_kept = cell_types[kept_idx]

type_to_int = {ct: i for i, ct in enumerate(sorted(unique_types))}
true_labels = np.array([type_to_int[ct] for ct in cell_types_kept])

ari = adjusted_rand_score(true_labels, assigns)
print(f'Adjusted Rand Index (MEK-Means vs ground truth): {ari:.3f}')
print(f'(0 = random assignment, 1 = perfect recovery)\n')

cm = confusion_matrix(true_labels, assigns)
print(f"{'':>12s}  " + '  '.join(f'Cluster {k}' for k in range(K_EM)))
for i, ct in enumerate(sorted(unique_types)):
    print(f'{ct:>12s}  ' + '  '.join(f'{cm[i,j]:9,d}' for j in range(K_EM)))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(K_EM))
ax.set_yticks(range(len(unique_types)))
ax.set_xticklabels([f'Cluster {k}' for k in range(K_EM)])
ax.set_yticklabels(sorted(unique_types))
ax.set_xlabel('MEK-Means cluster')
ax.set_ylabel('Ground-truth cell type')
ax.set_title(f'Cluster recovery  (ARI = {ari:.3f})')
for i in range(len(unique_types)):
    for j in range(K_EM):
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                color='white' if cm[i,j] > cm.max()*0.5 else 'black', fontsize=9)
fig.colorbar(im, ax=ax, label='Cell count')
fig.tight_layout()
plt.savefig('figures/pbmc_confusion_matrix.png', dpi=150)
plt.show()

---
## Summary

| Fast-path | Activates automatically? | Typical speedup |
|---|---|---|
| `e_step_2d` | Yes | scales with k × n_genes |
| `optimize_genes_2d` | Yes | scales with n_genes |

No flag changes required — both fast-paths fire inside `_e_step` and
`iterate_over_genes` whenever `monod_core` is installed.

In [ ]:
print('Fast-path availability')
print(f'  e_step_2d         : {hasattr(_mc, "e_step_2d")}')
print(f'  optimize_genes_2d : {hasattr(_mc, "optimize_genes_2d")}')
print(f'  Active in mminference: {mm_rust}')